# Breast Cancer Classification using ResNet50

Patient-disjoint 5-fold cross-validation on the BreaKHis histopathology dataset (400X magnification)


## Setup

In [ ]:
!nvidia-smi

Thu May 14 05:48:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |     546MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Google Drive path for dataset and results
BASE_DIR    = '/content/drive/MyDrive/Colab Notebooks/BreastCancerDetection/Dataset 2 breast cancer histopathology 400X'
RESULTS_DIR = '/content/drive/MyDrive/Colab Notebooks/BreastCancerDetection/results'

import os
assert os.path.isdir(BASE_DIR), f'Dataset not found: {BASE_DIR}'
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, csv, random, time
from collections import Counter, defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms, models

import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
BATCH_SIZE = 64 if USE_CUDA else 32
NUM_WORKERS = 2 if USE_CUDA else 0
PIN_MEMORY = USE_CUDA

SEED = 42
N_FOLDS = 5
SENSITIVITY_TARGET = 0.90
LR_CANDIDATES = [1e-5, 3e-5]
TUNING_EPOCHS = 15
FINAL_EPOCHS = 40
PATIENCE = 6

random.seed(SEED)
torch.manual_seed(SEED)

def _resolve(path):
    if not os.path.isabs(path):
        path = os.path.join(RESULTS_DIR, path)
    os.makedirs(os.path.dirname(path) or RESULTS_DIR, exist_ok=True)
    return path

name = torch.cuda.get_device_name(0) if USE_CUDA else 'CPU'
print(f'Device: {DEVICE} ({name}) | batch={BATCH_SIZE} workers={NUM_WORKERS} AMP={USE_CUDA}')

Device: cuda (NVIDIA A100-SXM4-40GB) | batch=64 workers=2 AMP=True


## Data module

In [ ]:
# BreaKHis filename pattern: SOB_<B|M>_<subtype>-<year>-<slide_id>-<mag>-<seq>.png
_FILENAME_RE = re.compile(
    r'^(?P<patient>SOB_[BM]_(?P<subtype>[A-Z]+)-\d+-[A-Z0-9]+)-\d+-\d+\.png$',
    re.IGNORECASE,
)

SUBTYPE_NAMES = {
    'A': 'Adenosis', 'F': 'Fibroadenoma', 'PT': 'Phyllodes Tumor',
    'TA': 'Tubular Adenoma', 'DC': 'Ductal Carcinoma', 'LC': 'Lobular Carcinoma',
    'MC': 'Mucinous Carcinoma', 'PC': 'Papillary Carcinoma',
}
CLASS_NAMES = ['benign', 'malignant']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def parse_filename(fname):
    m = _FILENAME_RE.match(fname)
    return (m.group('patient'), m.group('subtype').upper()) if m else None


def gather_samples(base_dir):
    samples = []
    for split in ('train', 'test'):  # pool both halves (raw split has patient leakage)
        for label, cls in enumerate(CLASS_NAMES):
            d = os.path.join(base_dir, split, cls)
            if not os.path.isdir(d):
                continue
            for fname in os.listdir(d):
                if not fname.lower().endswith('.png'):
                    continue
                parsed = parse_filename(fname)
                if parsed is None:
                    raise RuntimeError(f'Cannot parse filename {fname!r}')
                pid, subtype = parsed
                samples.append((os.path.join(d, fname), label, pid, subtype))
    if not samples:
        raise RuntimeError(f'No images found under {base_dir!r}')
    return samples


class HistopathologyDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label, *_ = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


class HistopathologyDataModule:
    class_names = CLASS_NAMES

    def __init__(self, train_samples, val_samples, test_samples,
                 batch_size=32, num_workers=0, pin_memory=False):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory

        train_tf = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
        eval_tf = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
        self.train_dataset = HistopathologyDataset(train_samples, train_tf)
        self.val_dataset = HistopathologyDataset(val_samples, eval_tf)
        self.test_dataset = HistopathologyDataset(test_samples, eval_tf)
        self.check_no_patient_leakage()

    @classmethod
    def kfold(cls, base_dir, k=5, val_ratio=0.15,
              batch_size=32, num_workers=0, pin_memory=False, seed=42):
        all_samples = gather_samples(base_dir)
        n_classes = len(cls.class_names)

        by_class_patient = {label: defaultdict(list) for label in range(n_classes)}
        for sample in all_samples:
            _, label, pid, *_ = sample
            by_class_patient[label][pid].append(sample)

        rng = random.Random(seed)
        shuffled = {}
        for label in range(n_classes):
            patients = sorted(by_class_patient[label].keys())
            rng.shuffle(patients)
            shuffled[label] = patients

        for fold_idx in range(k):
            train, val, test = [], [], []
            for label in range(n_classes):
                patients = shuffled[label]
                n = len(patients)
                test_start = fold_idx * n // k
                test_end = (fold_idx + 1) * n // k
                test_pids = patients[test_start:test_end]
                remaining = patients[:test_start] + patients[test_end:]
                n_val = max(1, int(round(len(remaining) * val_ratio)))
                val_pids, train_pids = remaining[:n_val], remaining[n_val:]
                for pid in train_pids:
                    train.extend(by_class_patient[label][pid])
                for pid in val_pids:
                    val.extend(by_class_patient[label][pid])
                for pid in test_pids:
                    test.extend(by_class_patient[label][pid])
            yield fold_idx, cls(train, val, test,
                                batch_size=batch_size, num_workers=num_workers,
                                pin_memory=pin_memory)

    def check_no_patient_leakage(self):
        def pids(ds): return {s[2] for s in ds.samples}
        tr, va, te = pids(self.train_dataset), pids(self.val_dataset), pids(self.test_dataset)
        for a_n, a, b_n, b in [('train',tr,'val',va), ('train',tr,'test',te), ('val',va,'test',te)]:
            overlap = a & b
            if overlap:
                raise RuntimeError(f'Patient leakage {a_n}<->{b_n}: {sorted(overlap)[:3]}')

    def train_loader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers, pin_memory=self.pin_memory)
    def val_loader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers, pin_memory=self.pin_memory)
    def test_loader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers, pin_memory=self.pin_memory)

    def pos_weight(self):
        counts = Counter(s[1] for s in self.train_dataset.samples)
        return counts[0] / counts[1]

    def describe(self):
        print(f'Classes: {self.class_names}')
        for name, ds in [('train', self.train_dataset), ('val', self.val_dataset), ('test', self.test_dataset)]:
            lc = Counter(s[1] for s in ds.samples)
            np_ = len({s[2] for s in ds.samples})
            parts = ' | '.join(f'{self.class_names[c]} {n}' for c, n in sorted(lc.items()))
            print(f'  {name:>5}: {len(ds):>4} images | {np_:>3} patients | {parts}')

## Model

In [ ]:
class ResNet50CancerModel(nn.Module):
    # Freeze early layers to avoid overfitting on this small dataset.
    # Only layer3, layer4 and the FC head are trained.

    FROZEN_PREFIXES = ('conv1', 'bn1', 'layer1', 'layer2')

    def __init__(self, num_classes=1, dropout=0.5):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )
        for name, p in self.backbone.named_parameters():
            if name.startswith(self.FROZEN_PREFIXES):
                p.requires_grad = False

    def forward(self, x):
        return self.backbone(x)

## Trainer

In [ ]:
class Trainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = next(model.parameters()).device
        self.use_amp = self.device.type == 'cuda'
        self.scaler = GradScaler(enabled=self.use_amp) if self.use_amp else None

    def _run_epoch(self, loader, train):
        self.model.train() if train else self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)
                if train:
                    self.optimizer.zero_grad()
                with autocast(device_type=self.device.type, enabled=self.use_amp):
                    logits = self.model(images).squeeze(1)
                    loss = self.criterion(logits, labels.float())
                if train:
                    if self.use_amp:
                        self.scaler.scale(loss).backward()
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        loss.backward()
                        self.optimizer.step()
                running_loss += loss.item() * images.size(0)
                preds = (torch.sigmoid(logits) > 0.5).float()
                correct += (preds == labels.float()).sum().item()
                total += labels.size(0)
        return running_loss / total, correct / total

    def fit(self, num_epochs=25, patience=5, checkpoint_path='best_model.pth'):
        checkpoint_path = _resolve(checkpoint_path)
        best_val_loss = float('inf')
        patience_counter = 0
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=num_epochs)
        run_start = time.time()
        epochs_run = 0
        for epoch in range(num_epochs):
            epoch_start = time.time()
            train_loss, train_acc = self._run_epoch(self.train_loader, train=True)
            val_loss, val_acc = self._run_epoch(self.val_loader, train=False)
            epoch_elapsed = time.time() - epoch_start
            epochs_run += 1
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            print(f'Epoch {epoch+1:>2}/{num_epochs} | '
                  f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
                  f'val loss {val_loss:.4f} acc {val_acc:.4f} | '
                  f'{epoch_elapsed:.1f}s')
            scheduler.step()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(self.model.state_dict(), checkpoint_path)
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break
        total_elapsed = time.time() - run_start
        print(f'Training time: {total_elapsed:.1f}s '
              f'({epochs_run} epochs, avg {total_elapsed/epochs_run:.1f}s/epoch)')
        # reload best weights
        self.model.load_state_dict(torch.load(checkpoint_path, map_location=self.device))
        return best_val_loss, history

    @torch.no_grad()
    def _infer(self, loader):
        self.model.eval()
        y_true, y_prob = [], []
        loss_total, n = 0.0, 0
        for images, labels in loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            with autocast(device_type=self.device.type, enabled=self.use_amp):
                logits = self.model(images).squeeze(1)
                loss = self.criterion(logits, labels.float())
            loss_total += loss.item() * images.size(0)
            y_true.extend(labels.tolist())
            y_prob.extend(torch.sigmoid(logits).tolist())
            n += labels.size(0)
        return y_true, y_prob, loss_total / n

    def tune_threshold(self, val_loader, target_sensitivity=0.95):
        """Find highest threshold that still meets the sensitivity target."""
        y_true, y_prob, _ = self._infer(val_loader)
        _, tpr, thresholds = roc_curve(y_true, y_prob)
        return float(thresholds[(tpr >= target_sensitivity).argmax()])

    @torch.no_grad()
    def evaluate_tta(self, test_loader, threshold=0.5):
        """Average predictions over 5 geometric augmentations (flips + 90deg rotation)."""
        self.model.eval()
        tta_ops = [
            lambda x: x,
            lambda x: torch.flip(x, dims=[-1]),
            lambda x: torch.flip(x, dims=[-2]),
            lambda x: torch.flip(x, dims=[-1, -2]),
            lambda x: torch.rot90(x, k=1, dims=[-2, -1]),
        ]
        start_time = time.time()
        y_true, y_prob = [], []
        loss_total, n = 0.0, 0
        for images, labels in test_loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            with autocast(device_type=self.device.type, enabled=self.use_amp):
                per_op_logits = [self.model(op(images)).squeeze(1) for op in tta_ops]
                id_loss = self.criterion(per_op_logits[0], labels.float())
            loss_total += id_loss.item() * images.size(0)
            probs = torch.stack([torch.sigmoid(l) for l in per_op_logits]).mean(dim=0)
            y_true.extend(labels.tolist())
            y_prob.extend(probs.tolist())
            n += labels.size(0)
        elapsed = time.time() - start_time
        y_pred = [int(p > threshold) for p in y_prob]
        accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
        return {
            'loss': loss_total / n, 'accuracy': accuracy,
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
            'runtime_seconds': elapsed, 'n_images': len(y_true),
        }


def hyperparameter_search(model_factory, train_loader, val_loader,
                          lrs, epochs=5, patience=5, pos_weight=None, device=None):
    if device is None:
        device = torch.device('cpu')
    pw_tensor = (torch.tensor([pos_weight], device=device)
                 if pos_weight is not None else None)
    results = []
    best_lr, best_loss = None, float('inf')
    for lr in lrs:
        print(f'\n--- Tuning LR = {lr} ---')
        model = model_factory().to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        trainer = Trainer(model, criterion, optimizer, train_loader, val_loader)
        val_loss, _ = trainer.fit(num_epochs=epochs, patience=patience,
                                  checkpoint_path='tuning_tmp.pth')
        results.append({'lr': lr, 'val_loss': val_loss})
        if val_loss < best_loss:
            best_loss = val_loss
            best_lr = lr
    return best_lr, best_loss, results

## Reporting helpers

In [ ]:
def plot_tuning_results(tuning_results, output_path='tuning_results.png'):
    output_path = _resolve(output_path)
    lrs = [r['lr'] for r in tuning_results]
    losses = [r['val_loss'] for r in tuning_results]
    plt.figure()
    plt.plot(lrs, losses, marker='o')
    plt.xscale('log'); plt.xlabel('Learning Rate'); plt.ylabel('Validation Loss')
    plt.title('HP Tuning: Validation Loss by Learning Rate')
    plt.grid(True); plt.tight_layout(); plt.savefig(output_path); plt.close()
    print(f'Saved tuning plot to {output_path}')


def plot_confusion_matrix(y_true, y_pred, class_names, output_path='confusion_matrix.png'):
    output_path = _resolve(output_path)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure()
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix'); plt.colorbar()
    ticks = list(range(len(class_names)))
    plt.xticks(ticks, class_names); plt.yticks(ticks, class_names)
    plt.ylabel('True label'); plt.xlabel('Predicted label')
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha='center',
                     color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout(); plt.savefig(output_path); plt.close()
    print(f'Saved confusion matrix to {output_path}')


def aggregate_by_patient(samples, y_prob, threshold):
    """Average patch probabilities per patient, then classify."""
    probs_by_pid = defaultdict(list)
    correct_by_pid = defaultdict(int)
    label_by_pid = {}
    for sample, prob in zip(samples, y_prob):
        _, label, pid, *_ = sample
        probs_by_pid[pid].append(prob)
        label_by_pid[pid] = label
        correct_by_pid[pid] += int(int(prob > threshold) == label)
    rows = []
    for pid in sorted(probs_by_pid):
        probs = probs_by_pid[pid]
        n = len(probs)
        rows.append({
            'pid': pid, 'true': label_by_pid[pid],
            'pred': int(sum(probs)/n > threshold), 'mean_prob': sum(probs)/n,
            'n_patches': n, 'n_correct': correct_by_pid[pid],
            'patient_score': correct_by_pid[pid] / n,
        })
    return rows


def print_subtype_breakdown(samples, y_true, y_pred, class_names):
    rows = defaultdict(lambda: {'n': 0, 'correct': 0, 'label': None})
    for sample, true, pred in zip(samples, y_true, y_pred):
        _, _, _, subtype = sample
        rows[subtype]['n'] += 1
        rows[subtype]['correct'] += int(true == pred)
        rows[subtype]['label'] = true
    print('\n' + '-' * 78)
    print('PER-SUBTYPE BREAKDOWN (patch-level)')
    print('-' * 78)
    print(f'{"code":<5} {"name":<22} {"class":<11} '
          f'{"#patch":>6} {"#correct":>8} {"#wrong":>7} {"recall":>7}')
    for c in sorted(rows, key=lambda c: (rows[c]['label'], c)):
        i = rows[c]
        print(f'{c:<5} {SUBTYPE_NAMES.get(c,c):<22} {class_names[i["label"]]:<11} '
              f'{i["n"]:>6} {i["correct"]:>8} {i["n"]-i["correct"]:>7} '
              f'{i["correct"]/i["n"] if i["n"] else 0:>7.4f}')


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = [int(p > threshold) for p in y_prob]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else 0.0
    spec = tn/(tn+fp) if (tn+fp) else 0.0
    ppv  = tp/(tp+fp) if (tp+fp) else 0.0
    f1   = 2*ppv*sens/(ppv+sens) if (ppv+sens) else 0.0
    acc  = (tp+tn)/len(y_true) if y_true else 0.0
    return {'threshold': threshold, 'sens': sens, 'spec': spec, 'ppv': ppv,
            'f1': f1, 'acc': acc, 'tn': int(tn), 'fp': int(fp),
            'fn': int(fn), 'tp': int(tp)}


def threshold_for_sensitivity(y_true, y_prob, target):
    _, tpr, thresholds = roc_curve(y_true, y_prob)
    eligible = tpr >= target
    if not eligible.any():
        return float(thresholds[-1])
    return float(thresholds[eligible.argmax()])


def threshold_for_youden(y_true, y_prob):
    """Youden's J = sensitivity + specificity - 1"""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return float(thresholds[(tpr - fpr).argmax()])


def print_full_report(results, patient_rows, threshold, class_names, roc_auc, ap,
                      samples=None):
    y_true, y_prob, y_pred = results['y_true'], results['y_prob'], results['y_pred']
    n_pos = sum(y_true)
    n_neg = len(y_true) - n_pos
    prevalence = n_pos / len(y_true) if y_true else 0.0

    print('\n' + '=' * 78)
    print('PATCH-LEVEL DISCRIMINATION (threshold-free)')
    print('=' * 78)
    print(f'Test set: {len(y_true)} patches ({class_names[0]} {n_neg}, '
          f'{class_names[1]} {n_pos}, prevalence {prevalence:.3f})')
    print(f'ROC AUC:           {roc_auc:.4f}')
    print(f'Average Precision: {ap:.4f}')
    print(f'Loss:              {results["loss"]:.4f}')
    rt = results.get('runtime_seconds')
    ni = results.get('n_images')
    if rt and ni:
        print(f'Inference:         {rt:.2f}s for {ni} images ({1000*rt/ni:.2f} ms/image)')

    # show metrics at a few different thresholds for comparison
    print('\n' + '-' * 78)
    print('OPERATING-POINT COMPARISON')
    print('-' * 78)
    candidates = [
        ('default 0.5', 0.5),
        ('Youden J',    threshold_for_youden(y_true, y_prob)),
        ('sens >= 0.95', threshold_for_sensitivity(y_true, y_prob, 0.95)),
        ('sens >= 0.90', threshold_for_sensitivity(y_true, y_prob, 0.90)),
        ('SELECTED',     threshold),
    ]
    print(f'{"point":<13} {"thresh":>7} {"sens":>7} {"spec":>7} '
          f'{"PPV":>7} {"F1":>7} {"acc":>7}  {"TN/FP/FN/TP":>15}')
    for label, t in candidates:
        m = metrics_at_threshold(y_true, y_prob, t)
        print(f'{label:<13} {m["threshold"]:>7.4f} {m["sens"]:>7.4f} '
              f'{m["spec"]:>7.4f} {m["ppv"]:>7.4f} {m["f1"]:>7.4f} '
              f'{m["acc"]:>7.4f}  {m["tn"]:>3}/{m["fp"]:>3}/{m["fn"]:>3}/{m["tp"]:>3}')

    print('\n' + '=' * 78)
    print(f'SELECTED OPERATING POINT @ threshold = {threshold:.4f}')
    print('=' * 78)
    print(f'Accuracy {results["accuracy"]:.4f}\n')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f'Sensitivity (recall, {class_names[1]}): {tp/(tp+fn):.4f}  [TP={tp}, FN={fn}]')
    print(f'Specificity (recall, {class_names[0]}):    {tn/(tn+fp):.4f}  [TN={tn}, FP={fp}]')

    if samples is not None:
        print_subtype_breakdown(samples, y_true, y_pred, class_names)

    # patient-level aggregation
    print('\n' + '=' * 86)
    print('PATIENT-LEVEL REPORT')
    print('=' * 86)
    print(f'Test set: {len(patient_rows)} patients\n')
    print(f'{"patient":<25} {"true":<11} {"#img":>5}  {"mean p":>7}  '
          f'{"pred":<11} {"match":<6} {"#corr":>5}  {"score":>7}')
    print('-' * 86)
    verdict_correct = 0
    verdict_by_class = defaultdict(lambda: [0, 0])
    score_by_class = defaultdict(list)
    for r in patient_rows:
        ok = r['pred'] == r['true']
        print(f'{r["pid"]:<25} {class_names[r["true"]]:<11} '
              f'{r["n_patches"]:>5}  {r["mean_prob"]:>7.4f}  '
              f'{class_names[r["pred"]]:<11} {"OK" if ok else "WRONG":<6} '
              f'{r["n_correct"]:>5}  {r["patient_score"]:>7.4f}')
        verdict_correct += int(ok)
        verdict_by_class[r['true']][1] += 1
        verdict_by_class[r['true']][0] += int(ok)
        score_by_class[r['true']].append(r['patient_score'])
    print('-' * 86)
    n = len(patient_rows)
    print(f'\nAggregate-then-decide accuracy: {verdict_correct}/{n} = {100*verdict_correct/n:.1f}%')
    for c in (0, 1):
        nc, nt = verdict_by_class[c]
        if nt:
            print(f'    {class_names[c]:<11}: {nc}/{nt} = {100*nc/nt:.1f}%')
    all_scores = [r['patient_score'] for r in patient_rows]
    print(f'\nBreaKHis Patient Score (image-level, averaged per patient):')
    print(f'  Recognition Rate: {sum(all_scores)/n:.4f}')
    for c in (0, 1):
        sc = score_by_class[c]
        if sc:
            print(f'    {class_names[c]:<11}: {sum(sc)/len(sc):.4f} '
                  f'(mean over {len(sc)} patient{"s" if len(sc)!=1 else ""})')


def print_kfold_summary(fold_results, class_names):
    k = len(fold_results)
    print('\n' + '#' * 78)
    print(f'K-FOLD AGGREGATE SUMMARY ({k} folds)')
    print('#' * 78)
    print(f'\n{"fold":<5} {"thresh":>7} {"loss":>7} {"acc":>7} '
          f'{"AUC":>7} {"AP":>7} {"sens":>7} {"spec":>7}')
    sens_list, spec_list = [], []
    for fr in fold_results:
        m = metrics_at_threshold(fr['y_true'], fr['y_prob'], fr['threshold'])
        sens_list.append(m['sens']); spec_list.append(m['spec'])
        print(f'{fr["fold_idx"]+1:<5} {fr["threshold"]:>7.4f} '
              f'{fr["loss"]:>7.4f} {fr["accuracy"]:>7.4f} '
              f'{fr["roc_auc"]:>7.4f} {fr["avg_precision"]:>7.4f} '
              f'{m["sens"]:>7.4f} {m["spec"]:>7.4f}')
    def mean_sd(xs):
        m = sum(xs)/len(xs)
        return m, (sum((x-m)**2 for x in xs)/max(len(xs)-1,1))**0.5
    print('\nMean +/- SD across folds:')
    for name, vals in [('ROC AUC', [f['roc_auc'] for f in fold_results]),
                       ('AP', [f['avg_precision'] for f in fold_results]),
                       ('accuracy', [f['accuracy'] for f in fold_results]),
                       ('sensitivity', sens_list), ('specificity', spec_list)]:
        m, s = mean_sd(vals)
        print(f'  {name:<12}: {m:.4f} +/- {s:.4f}')
    pooled_t = [t for f in fold_results for t in f['y_true']]
    pooled_p = [p for f in fold_results for p in f['y_prob']]
    fpr_p, tpr_p, _ = roc_curve(pooled_t, pooled_p)
    print(f'\nPooled ({len(pooled_t)} patches, {sum(pooled_t)} malignant):')
    print(f'  ROC AUC:           {auc(fpr_p, tpr_p):.4f}')
    print(f'  Average Precision: {average_precision_score(pooled_t, pooled_p):.4f}')
    all_pr = [r for f in fold_results for r in f['patient_rows']]
    correct = sum(1 for r in all_pr if r['pred'] == r['true'])
    print(f'\nPatient-level pooled ({len(all_pr)} patients):')
    print(f'  Accuracy: {correct}/{len(all_pr)} = {100*correct/len(all_pr):.1f}%')
    pooled_samples = [s for f in fold_results for s in f['samples']]
    pooled_pred = [p for f in fold_results for p in f['y_pred']]
    print_subtype_breakdown(pooled_samples, pooled_t, pooled_pred, class_names)


def save_kfold_summary(fold_results, class_names, prefix='summary/'):
    """Save all the summary plots and CSV to one folder."""
    from matplotlib.patches import Patch

    pooled_true = [t for f in fold_results for t in f['y_true']]
    pooled_prob = [p for f in fold_results for p in f['y_prob']]
    pooled_pred = [p for f in fold_results for p in f['y_pred']]
    fpr_p, tpr_p, _ = roc_curve(pooled_true, pooled_prob)
    pooled_auc = auc(fpr_p, tpr_p)
    pooled_ap = average_precision_score(pooled_true, pooled_prob)

    # pooled ROC with per-fold overlays
    plt.figure()
    for fr in fold_results:
        fp, tp, _ = roc_curve(fr['y_true'], fr['y_prob'])
        plt.plot(fp, tp, alpha=0.25, linewidth=1,
                 label=f'Fold {fr["fold_idx"]+1} (AUC={fr["roc_auc"]:.3f})')
    plt.plot(fpr_p, tpr_p, color='navy', linewidth=2,
             label=f'Pooled (AUC={pooled_auc:.4f})')
    plt.plot([0,1], [0,1], '--', color='gray')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('Pooled ROC Curve'); plt.legend(loc='lower right', fontsize=8)
    plt.grid(True); plt.tight_layout()
    plt.savefig(_resolve(prefix + 'pooled_roc_curve.png')); plt.close()

    # pooled confusion matrix
    plot_confusion_matrix(pooled_true, pooled_pred, class_names,
                          output_path=prefix + 'pooled_confusion_matrix.png')

    # per-fold comparison
    k = len(fold_results)
    x = range(k); w = 0.25
    plt.figure(figsize=(10, 5))
    plt.bar([i-w for i in x], [f['roc_auc'] for f in fold_results], w,
            label='ROC AUC', color='steelblue')
    plt.bar(x, [f['avg_precision'] for f in fold_results], w,
            label='Avg Precision', color='darkorange')
    plt.bar([i+w for i in x], [f['accuracy'] for f in fold_results], w,
            label='Accuracy', color='seagreen')
    plt.xticks(x, [f'Fold {f["fold_idx"]+1}' for f in fold_results])
    plt.ylabel('Score'); plt.title('Per-Fold Performance')
    plt.legend(); plt.ylim(0.6, 1.05); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
    plt.savefig(_resolve(prefix + 'per_fold_comparison.png')); plt.close()

    # subtype recall
    pooled_samples = [s for f in fold_results for s in f['samples']]
    st = defaultdict(lambda: {'n': 0, 'correct': 0, 'label': None})
    for sample, true, pred in zip(pooled_samples, pooled_true, pooled_pred):
        _, _, _, subtype = sample
        st[subtype]['n'] += 1
        st[subtype]['correct'] += int(true == pred)
        st[subtype]['label'] = true
    codes = sorted(st, key=lambda c: (st[c]['label'], c))
    names = [SUBTYPE_NAMES.get(c, c) for c in codes]
    recalls = [st[c]['correct']/st[c]['n'] for c in codes]
    colors = ['#4a90d9' if st[c]['label']==0 else '#d94a4a' for c in codes]
    plt.figure(figsize=(10, 5))
    bars = plt.bar(names, recalls, color=colors)
    plt.ylabel('Patch-Level Recall'); plt.title('Per-Subtype Recall (pooled)')
    plt.xticks(rotation=30, ha='right'); plt.ylim(0, 1.1)
    for bar, val in zip(bars, recalls):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', fontsize=9)
    plt.legend(handles=[Patch(color='#4a90d9', label='Benign'),
                        Patch(color='#d94a4a', label='Malignant')], loc='lower right')
    plt.grid(axis='y', alpha=0.3); plt.tight_layout()
    plt.savefig(_resolve(prefix + 'subtype_recall.png')); plt.close()

    # combined training curves (all folds on one plot)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for fr in fold_results:
        h = fr.get('history')
        if not h:
            continue
        lbl = f'Fold {fr["fold_idx"]+1}'
        ax1.plot(h['train_loss'], '-', alpha=0.4, label=lbl+' train')
        ax1.plot(h['val_loss'], '--', alpha=0.7, label=lbl+' val')
        ax2.plot(h['train_acc'], '-', alpha=0.4, label=lbl+' train')
        ax2.plot(h['val_acc'], '--', alpha=0.7, label=lbl+' val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss'); ax1.legend(fontsize=7, ncol=2); ax1.grid(True, alpha=0.3)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title('Training & Validation Accuracy'); ax2.legend(fontsize=7, ncol=2); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(_resolve(prefix + 'training_curves.png')); plt.close()

    # summary csv
    csv_path = _resolve(prefix + 'fold_summary.csv')
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['fold','threshold','loss','accuracy','roc_auc','avg_precision','sensitivity','specificity'])
        for fr in fold_results:
            m = metrics_at_threshold(fr['y_true'], fr['y_prob'], fr['threshold'])
            w.writerow([fr['fold_idx']+1, f'{fr["threshold"]:.4f}', f'{fr["loss"]:.4f}',
                        f'{fr["accuracy"]:.4f}', f'{fr["roc_auc"]:.4f}',
                        f'{fr["avg_precision"]:.4f}', f'{m["sens"]:.4f}', f'{m["spec"]:.4f}'])
        w.writerow(['pooled','','','', f'{pooled_auc:.4f}', f'{pooled_ap:.4f}','',''])

    print(f'Saved summary files to {_resolve(prefix)}')


# --- confidence triage (for the CDSS framing) ---

def triage_predictions(y_prob, threshold, margin=0.15):
    """Bin each prediction into confident benign / uncertain / confident malignant."""
    tiers = []
    for p in y_prob:
        if p < (threshold - margin):
            tiers.append('confident_benign')
        elif p > (threshold + margin):
            tiers.append('confident_malignant')
        else:
            tiers.append('uncertain_review')
    return tiers


def print_triage_report(y_true, y_prob, threshold, class_names,
                        margin=0.15, prefix='', save=False):
    tiers = triage_predictions(y_prob, threshold, margin)
    n = len(y_true)
    print('\n' + '=' * 78)
    print('CLINICAL DECISION SUPPORT TRIAGE')
    print('=' * 78)
    print(f'Threshold: {threshold:.4f}, uncertainty margin: +/-{margin}')
    print(f'  Confident benign:    prob < {threshold-margin:.4f}')
    print(f'  Uncertain (review):  {threshold-margin:.4f} <= prob <= {threshold+margin:.4f}')
    print(f'  Confident malignant: prob > {threshold+margin:.4f}\n')
    print(f'{"tier":<25} {"count":>6} {"% of total":>10} {"accuracy":>9}')
    print('-' * 55)
    tier_data = []
    for tn in ['confident_benign', 'uncertain_review', 'confident_malignant']:
        idx = [i for i, t in enumerate(tiers) if t == tn]
        if not idx:
            tier_data.append((tn, 0, 0.0, 0.0))
            print(f'{tn:<25} {"0":>6} {"0.0%":>10} {"N/A":>9}')
            continue
        correct = sum(1 for i in idx if int(y_prob[i] > threshold) == y_true[i])
        acc = correct / len(idx)
        pct = 100 * len(idx) / n
        print(f'{tn:<25} {len(idx):>6} {pct:>9.1f}% {acc:>9.4f}')
        tier_data.append((tn, len(idx), pct, acc))
    unc = tiers.count('uncertain_review')
    conf = n - unc
    conf_acc = (sum(1 for i in range(n) if tiers[i]!='uncertain_review'
                    and int(y_prob[i]>threshold)==y_true[i]) / conf) if conf else 0
    print(f'\nConfident: {conf}/{n} ({100*conf/n:.1f}%) accuracy {conf_acc:.4f}')
    print(f'Deferred:  {unc}/{n} ({100*unc/n:.1f}%)')

    if save and prefix:
        csv_path = _resolve(prefix + 'triage_summary.csv')
        with open(csv_path, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['tier','count','pct','accuracy'])
            for tn, cnt, pct, acc in tier_data:
                w.writerow([tn, cnt, f'{pct:.1f}', f'{acc:.4f}'])
        labels = ['Confident\nBenign', 'Uncertain\n(Review)', 'Confident\nMalignant']
        counts = [d[1] for d in tier_data]
        colors = ['#4a90d9', '#f5a623', '#d94a4a']
        plt.figure(figsize=(8, 5))
        bs = plt.bar(labels, counts, color=colors, edgecolor='white', linewidth=1.5)
        for b, (_, cnt, pct, acc) in zip(bs, tier_data):
            if cnt > 0:
                plt.text(b.get_x()+b.get_width()/2, b.get_height()+2,
                         f'{cnt} ({pct:.0f}%)\nacc: {acc:.2f}', ha='center', fontsize=9)
        plt.ylabel('Number of Patches')
        plt.title(f'CDSS Triage (threshold={threshold:.3f}, margin=+/-{margin})')
        plt.tight_layout()
        plt.savefig(_resolve(prefix + 'triage_chart.png')); plt.close()
        print(f'Saved triage files to {_resolve(prefix)}')

## Build k-fold splits

In [ ]:
folds = list(HistopathologyDataModule.kfold(
    base_dir=BASE_DIR, k=N_FOLDS, val_ratio=0.15,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, seed=SEED,
))
_, tune_dm = folds[0]
tune_dm.describe()
pos_weight = tune_dm.pos_weight()
print(f'pos_weight (fold 1 train) = {pos_weight:.4f}')

Classes: ['benign', 'malignant']
  train: 1232 images |  57 patients | benign 416 | malignant 816
    val:  192 images |  10 patients | benign 55 | malignant 137
   test:  269 images |  15 patients | benign 76 | malignant 193
pos_weight (fold 1 train) = 0.5098


## Hyperparameter tuning

In [ ]:
pipeline_start = time.time()
tuning_start = time.time()
best_lr, best_val_loss, tuning_results = hyperparameter_search(
    model_factory=ResNet50CancerModel,
    train_loader=tune_dm.train_loader(), val_loader=tune_dm.val_loader(),
    lrs=LR_CANDIDATES, epochs=TUNING_EPOCHS, patience=5,
    pos_weight=pos_weight, device=DEVICE,
)
plot_tuning_results(tuning_results)
print(f'Best LR = {best_lr}, best val loss = {best_val_loss:.4f}')
print(f'HP tuning time: {time.time() - tuning_start:.1f}s')


--- Tuning LR = 1e-05 ---
Epoch  1/15 | train loss 0.4547 acc 0.6071 | val loss 0.3547 acc 0.8854 | 243.6s
Epoch  2/15 | train loss 0.3972 acc 0.7248 | val loss 0.2823 acc 0.9375 | 20.5s
Epoch  3/15 | train loss 0.3374 acc 0.8003 | val loss 0.2128 acc 0.9167 | 22.0s
Epoch  4/15 | train loss 0.2827 acc 0.8287 | val loss 0.1819 acc 0.8958 | 22.4s
Epoch  5/15 | train loss 0.2396 acc 0.8669 | val loss 0.1597 acc 0.9010 | 22.2s
Epoch  6/15 | train loss 0.2202 acc 0.8750 | val loss 0.1591 acc 0.8906 | 22.3s
Epoch  7/15 | train loss 0.1955 acc 0.8961 | val loss 0.1639 acc 0.8854 | 22.4s
Epoch  8/15 | train loss 0.1628 acc 0.9188 | val loss 0.1686 acc 0.8854 | 20.3s
Epoch  9/15 | train loss 0.1555 acc 0.9229 | val loss 0.1727 acc 0.8750 | 20.4s
Epoch 10/15 | train loss 0.1501 acc 0.9237 | val loss 0.1698 acc 0.8854 | 20.5s
Epoch 11/15 | train loss 0.1330 acc 0.9334 | val loss 0.1651 acc 0.8854 | 20.3s
Early stopping at epoch 11
Training time: 458.5s (11 epochs, avg 41.7s/epoch)

--- Tuning LR

## Cross-validation training

In [ ]:
def train_fold(fold_idx, dm, best_lr):
    class_names = dm.class_names
    pos_weight = dm.pos_weight()

    print('\n' + '=' * 78)
    print(f'FOLD {fold_idx + 1}/{N_FOLDS}')
    print('=' * 78)
    dm.describe()
    print(f'pos_weight = {pos_weight:.4f}')

    model = ResNet50CancerModel().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], device=DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=best_lr, weight_decay=1e-4)
    trainer = Trainer(model, criterion, optimizer,
                      dm.train_loader(), dm.val_loader())

    train_start = time.time()
    _, history = trainer.fit(
        num_epochs=FINAL_EPOCHS, patience=PATIENCE,
        checkpoint_path='/tmp/fold_checkpoint.pth')
    print(f'Fold {fold_idx+1} training time: {time.time()-train_start:.1f}s')

    # Threshold: highest that keeps sensitivity >= target on val set
    threshold = trainer.tune_threshold(dm.val_loader(),
                                       target_sensitivity=SENSITIVITY_TARGET)
    print(f'Operating threshold: {threshold:.4f} '
          f'(targeting >={SENSITIVITY_TARGET:.0%} sensitivity on val)')

    results = trainer.evaluate_tta(dm.test_loader(), threshold=threshold)
    print(f'TTA accuracy: {results["accuracy"]:.4f} (loss {results["loss"]:.4f})')

    # Compute AUC and AP (no per-fold plots)
    fpr, tpr, _ = roc_curve(results['y_true'], results['y_prob'])
    roc_auc = auc(fpr, tpr)
    ap = average_precision_score(results['y_true'], results['y_prob'])
    print(f'ROC AUC: {roc_auc:.4f} | AP: {ap:.4f}')

    patient_rows = aggregate_by_patient(
        dm.test_dataset.samples, results['y_prob'], threshold)

    print_full_report(results, patient_rows, threshold, class_names,
                      roc_auc, ap, samples=dm.test_dataset.samples)

    # CDSS triage (console only, no per-fold file output)
    print_triage_report(results['y_true'], results['y_prob'], threshold,
                        class_names, margin=0.15, save=False)

    return {
        'fold_idx': fold_idx, 'threshold': threshold,
        'loss': results['loss'], 'accuracy': results['accuracy'],
        'roc_auc': roc_auc, 'avg_precision': ap,
        'y_true': results['y_true'], 'y_pred': results['y_pred'],
        'y_prob': results['y_prob'],
        'patient_rows': patient_rows, 'samples': dm.test_dataset.samples,
        'history': history,
    }


all_fold_results = []
for fold_idx, dm in folds:
    all_fold_results.append(train_fold(fold_idx, dm, best_lr))


FOLD 1/5
Classes: ['benign', 'malignant']
  train: 1232 images |  57 patients | benign 416 | malignant 816
    val:  192 images |  10 patients | benign 55 | malignant 137
   test:  269 images |  15 patients | benign 76 | malignant 193
pos_weight = 0.5098
Epoch  1/40 | train loss 0.4577 acc 0.4919 | val loss 0.3941 acc 0.8073 | 20.7s
Epoch  2/40 | train loss 0.3915 acc 0.6778 | val loss 0.2977 acc 0.8698 | 20.4s
Epoch  3/40 | train loss 0.3323 acc 0.7922 | val loss 0.2246 acc 0.8750 | 20.6s
Epoch  4/40 | train loss 0.2722 acc 0.8498 | val loss 0.1952 acc 0.8802 | 20.6s
Epoch  5/40 | train loss 0.2276 acc 0.8709 | val loss 0.1660 acc 0.9010 | 20.6s
Epoch  6/40 | train loss 0.1888 acc 0.9058 | val loss 0.1838 acc 0.8750 | 20.6s
Epoch  7/40 | train loss 0.1714 acc 0.9140 | val loss 0.1889 acc 0.8594 | 20.4s
Epoch  8/40 | train loss 0.1441 acc 0.9253 | val loss 0.2021 acc 0.8750 | 20.5s
Epoch  9/40 | train loss 0.1198 acc 0.9310 | val loss 0.2208 acc 0.8542 | 20.3s
Epoch 10/40 | train loss

## Aggregate report

In [ ]:
print_kfold_summary(all_fold_results, class_names=folds[0][1].class_names)
save_kfold_summary(all_fold_results, class_names=folds[0][1].class_names)

# Pooled CDSS triage (saved to summary/)
pooled_true = [t for fr in all_fold_results for t in fr['y_true']]
pooled_prob = [p for fr in all_fold_results for p in fr['y_prob']]
pooled_threshold = sorted([fr['threshold'] for fr in all_fold_results])[len(all_fold_results)//2]
print_triage_report(pooled_true, pooled_prob, pooled_threshold,
                    class_names=folds[0][1].class_names, margin=0.15,
                    prefix='summary/', save=True)

print(f'\nTotal pipeline time: {time.time() - pipeline_start:.1f}s')


##############################################################################
K-FOLD AGGREGATE SUMMARY (5 folds)
##############################################################################

fold   thresh    loss     acc     AUC      AP    sens    spec
1      0.4209  0.1568  0.9591  0.9901  0.9959  0.9896  0.8816
2      0.5054  0.3958  0.7978  0.8639  0.9029  0.9355  0.5903
3      0.4121  0.1730  0.9139  0.9651  0.9839  0.9252  0.8868
4      0.5132  0.3688  0.8146  0.8979  0.9583  0.7886  0.8727
5      0.3765  0.1031  0.9568  0.9907  0.9951  0.9746  0.9189

Mean +/- SD across folds:
  ROC AUC     : 0.9416 +/- 0.0576
  AP          : 0.9672 +/- 0.0390
  accuracy    : 0.8884 +/- 0.0774
  sensitivity : 0.9227 +/- 0.0796
  specificity : 0.8301 +/- 0.1352

Pooled (1693 patches, 1146 malignant):
  ROC AUC:           0.9218
  Average Precision: 0.9594

Patient-level pooled (82 patients):
  Accuracy: 77/82 = 93.9%

----------------------------------------------------------------------------